# Fine-tuning gain versus domain shift (Fig. A10)

Relates the gain from fine-tuning the multi-region model (`3.2 Pooled_model`) in each target region to the domain shift between the pooled source data and that region (Sinkhorn distance of the climate inputs, the topographic inputs and the geographic location).

| Paper figure | Content | Saved as (in `figures/paperTF/`) |
|---|---|---|
| Fig. A10 | Annual RMSE gain of full fine-tuning versus Sinkhorn distance | `appendix/A10_ds_pooled_gain.png` |

**Steps**
1. Load the stake datasets and monthly inputs, and split Central Asia, Alaska and Italy/Austria into sub-regions.
2. Load the multi-region model and its fine-tuned models, and compute the fine-tuning gain per target region.
3. Compute (or load) the domain shift between the pooled source data and each target region.
4. Plot the gain against the domain shift.

**Requires** the multi-region and fine-tuned models of `3.2 Pooled_model` (`cfg.dataPath/cache/TF_Europe/FD_MODEL/models/`). The domain shifts are cached in `FD_MODEL/domain_shifts_pooled_cache.pkl`. The functions used here live in `scripts/dataset/pooled_distance.py`, `scripts/models/pooled_model.py` and `scripts/plotting/domain_shift.py`.

## Setting up:

In [ ]:
import os

# Silence KeOps (used by geomloss for Sinkhorn distances) before it is imported
os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

import torch

# Initialise CUDA before the other imports: changing this order can change the
# GPU results slightly (~1e-7)
if torch.cuda.is_available():
    torch.cuda.init()

import sys
import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import massbalancemachine as mbm

sys.path.append(os.path.join(os.getcwd(), '../../'))  # Add root of repo to import MBM
# Shared config (paths, MONTHLY_COLS, STATIC_COLS, ...), model settings
# (PARAMS_TRANSFORMER, PARAMS_FT, N_EPOCHS, MODEL_DATE) and helpers
from papers.Transformer_TF.scripts.config_TF_Europe import *
from papers.Transformer_TF.scripts.config_models import *
from papers.Transformer_TF.scripts.dataset import *
from papers.Transformer_TF.scripts.plotting import *
from papers.Transformer_TF.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

# Call config file (data paths, seed, etc.)
cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)  # reproducibility
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

# Use the GPU if one is visible
device = torch.device("cuda" if torch.cuda.device_count() > 0 else "cpu")
print(f"Device: {device}")

# Cache folder of the multi-region experiments (datasets, models, domain shifts)
BASE_DIR = Path(cfg.dataPath) / path_cache / "FD_MODEL"
CACHE_DIR = BASE_DIR / "datasets"
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"Base directory for data: {BASE_DIR}")

os.makedirs("figures/paperTF/appendix", exist_ok=True)

## Data:
With `run_flag_by_code = False` (default), the monthly data are loaded from `WGMS/CrossRegional/monthly_cache/` instead of being recomputed.

In [ ]:
SOURCE_REGIONS = ['CH', 'NOR', 'ISL', "FR"]  # training regions of the multi-region model
TARGET_REGIONS = ['IT_AT', 'SJM', 'CENTRALASIA', 'ALA']
TARGET_REGIONS_SUB = ['IT', 'AT', 'SJM', 'ALA', 'CA_3', 'CA_4']

data = load_pooled_experiment_data(cfg, SOURCE_REGIONS, TARGET_REGIONS,
                                   TARGET_REGIONS_SUB)
monthly_cache = data["monthly_cache"]
res_xreg_by_source = data["res_xreg_by_source"]
months_head_pad = data["months_head_pad"]
months_tail_pad = data["months_tail_pad"]

## Fine-tuning gain:
Same models and fine-tuning/test splits as Fig. 6 (`3.2 Pooled_model`): full fine-tuning on 10% of the measurements of each target region, `N_REPEATS` repeats. The gain is the zero-shot minus the fine-tuned RMSE on the remaining 90% (positive = fine-tuning helps), averaged over the repeats.

In [ ]:
# FT_FRAC and the number of repeats are set in scripts/config_models.py
N_REPEATS = FT_N_REPEATS

assets_full = build_pooled_assets(
    res_xreg_by_source,
    SOURCE_REGIONS,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    months_head_pad=months_head_pad,
    months_tail_pad=months_tail_pad,
    cache_path=CACHE_DIR / "assets_foundation_100pct.joblib",
)
model_pooled, ds_pooled_scaler = load_pooled_model(
    cfg,
    device,
    assets_full,
    params=PARAMS_TRANSFORMER,
    models_dir=BASE_DIR / "models" / "foundation",
    epochs=N_EPOCHS,
    date=MODEL_DATE,
)

ft_assets_id = build_ft_assets_id_split(
    monthly_cache,
    TARGET_REGIONS_SUB,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    cache_dir=CACHE_DIR,
    ft_frac=FT_FRAC,
    n_repeats=N_REPEATS,
)
ft_models_id = finetune_id_split_models(
    cfg,
    device,
    model_pooled,
    ds_pooled_scaler,
    ft_assets_id,
    TARGET_REGIONS_SUB,
    models_dir_ft=BASE_DIR / "models" / "finetuned",
    params=PARAMS_TRANSFORMER,
    ft_params=PARAMS_FT,
    date=MODEL_DATE,
    strategies=("full",),
)

In [ ]:
df_ft_metrics_id, _ = ft_metrics_id_split(cfg,
                                          device,
                                          model_pooled,
                                          ds_pooled_scaler,
                                          ft_assets_id,
                                          ft_models_id,
                                          TARGET_REGIONS_SUB,
                                          strategies=("full",))

# {region: {"ft_gain_full": {"RMSE_annual_gain": ..., "RMSE_annual_gain_std": ...}}}
ft_gain = compute_ft_gain(df_ft_metrics_id, strategy="full")

## Domain shift:
Sinkhorn distances between the pooled training data of all source regions and all data of each target region. Inputs are standardised with scalers fitted on the source data; the geographic location is used as unit-sphere coordinates. Loaded from the cache unless `RECOMPUTE_SHIFTS_POOLED = True`.

In [ ]:
RECOMPUTE_SHIFTS_POOLED = False

all_shifts_pooled = compute_pooled_domain_shifts(
    cfg,
    res_xreg_by_source,
    monthly_cache,
    SOURCE_REGIONS,
    TARGET_REGIONS_SUB,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cache_path=BASE_DIR / "domain_shifts_pooled_cache.pkl",
    recompute=RECOMPUTE_SHIFTS_POOLED,
)

## Fig. A10: fine-tuning gain versus domain shift

In [ ]:
fig = plot_pooled_performance_vs_shift(
    all_shifts_pooled,
    ft_gain,
    metric="RMSE_annual_gain",
    model_label="ft_gain_full",
    distance_metrics=["D_sinkhorn_climate", "D_sinkhorn_topo", "D_sinkhorn_geo"],
    suptitle='Fine-tuning gain versus Sinkhorn distance')
fig.savefig('figures/paperTF/appendix/A10_ds_pooled_gain.png')
plt.show()